In [36]:
# from logit_model import *
import statsmodels.api as sm
import matplotlib.pyplot as plt
from tqdm import tqdm
from statsmodels.stats.outliers_influence import variance_inflation_factor


In [37]:
import sys
import os

# Add parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(parent_dir)

In [38]:
import pandas as pd

In [39]:
import package_files.benefits_defns as benefits_defns

In [40]:
import importlib 
importlib.reload(benefits_defns)
from package_files.benefits_defns import *

In [41]:
benefits4

['EDU_ASSISTANCE',
 'WLB',
 'PAID_LEAVE',
 'HEALTH_WELLBEING',
 'PARENTAL_LEAVE',
 'CULTURE']

In [31]:
from package_files.logit_model import *

In [32]:
path = '../data/us_10m_nointernship_ai_skills_benefits.parquet.gzip'

In [33]:
if path[-3:] == 'csv:':
    df = pd.read_csv(path)
elif path[-3:] == 'zip':
    df = pd.read_parquet(path)

In [34]:
df.columns

Index(['ID', 'CITY_NAME', 'COUNTY_NAME', 'STATE_NAME', 'NAICS_2022_2',
       'NAICS_2022_2_NAME', 'MIN_EDULEVELS_NAME', 'EMPLOYMENT_TYPE',
       'EMPLOYMENT_TYPE_NAME', 'IS_INTERNSHIP', 'REMOTE_TYPE',
       'REMOTE_TYPE_NAME', 'SKILLS', 'SKILLS_NAME', 'POSTED',
       'LAST_UPDATED_DATE', 'ONET', 'ONET_NAME', 'ONET_2019', 'ONET_2019_NAME',
       'SOC_2021_2_NAME', 'SOC_2021_2', 'MIN_YEARS_EXPERIENCE', 'COMPANY',
       'COMPANY_NAME', 'COMPANY_RAW', 'AI ROLE', 'wfh_wham_prob', 'wfh_wham',
       'YEAR', 'EDU_ASSISTANCE', 'WLB', 'PAID_LEAVE', 'HEALTH_WELLBEING',
       'PARENTAL_LEAVE', 'CULTURE', 'EXPERIENCE_BUCKET', 'SALARY',
       'LOG_SALARY'],
      dtype='object')

In [15]:
occupations_select = ['Architecture and Engineering Occupations','Arts, Design, Entertainment, Sports, and Media Occupations','Business and Financial Operations Occupations','Community and Social Service Occupations','Computer and Mathematical Occupations',
'Educational Instruction and Library Occupations',
'Healthcare Practitioners and Technical Occupations',
'Legal Occupations',
'Life, Physical, and Social Science Occupations',
'Management Occupations', 
'Office and Administrative Support Occupations',
'Personal Care and Service Occupations', 'Production Occupations',
'Sales and Related Occupations',
'Transportation and Material Moving Occupations',
]

In [16]:
df_select = df[df[occupation].isin(occupations_select)]

In [17]:
years = df['YEAR'].unique()

In [21]:
years

array([2019, 2021, 2022, 2020, 2018, 2023], dtype=int32)

In [20]:
df_select[df_select[occupation] == 'Architecture and Engineering Occupations'].groupby('YEAR').count()

,ID,CITY_NAME,COUNTY_NAME,STATE_NAME,NAICS_2022_2,NAICS_2022_2_NAME,MIN_EDULEVELS_NAME,EMPLOYMENT_TYPE,EMPLOYMENT_TYPE_NAME,IS_INTERNSHIP,...,wfh_wham,EDU_ASSISTANCE,WLB,LEAVE,HEALTH_WELLBEING,PARENTAL_LEAVE,CULTURE,EXPERIENCE_BUCKET,SALARY,LOG_SALARY
YEAR,,,,,,,,,,,,,,,,,,,,,
2018,40569,40569,40543,40569,40569,40569,40569,40569,40569,40569,...,15,40569,40569,40569,40569,40569,40569,40569,4994,4994
2019,42432,42432,42409,42432,42432,42432,42432,42432,42432,42432,...,41213,42432,42432,42432,42432,42432,42432,42432,4920,4920
2020,33522,33522,33514,33522,33522,33522,33522,33522,33522,33522,...,33182,33522,33522,33522,33522,33522,33522,33522,5104,5104
2021,42098,42098,42078,42098,42098,42098,42098,42098,42098,42098,...,41963,42098,42098,42098,42098,42098,42098,42098,7372,7372
2022,57103,57103,57083,57103,57103,57103,57103,57103,57103,57103,...,56350,57103,57103,57103,57103,57103,57103,57103,10769,10769
2023,46241,46241,46235,46241,46241,46241,46241,46241,46241,46241,...,45933,46241,46241,46241,46241,46241,46241,46241,15691,15691


In [36]:
def get_occ_year_data(df, occ, year):
    """
    Filter the DataFrame to include only rows with the specified occupation and year.

    Args:
        df (pd.DataFrame): The input DataFrame containing the data.
        occ (str): The occupation to filter by.
        year (int): The year to filter by.

    Returns:
        pd.DataFrame: A new DataFrame containing only the rows that match the specified occupation and year.
    """
    df_occ_year = df[(df[occupation] == occ) & (df['YEAR'] == year)].copy()
    return df_occ_year

In [39]:
models_dict = {}
for benefit in benefits4[2:]:
    print(benefit)
    for occ, year in tqdm([(occ, year) for occ in occupations_select for year in years]):
        print(occ, year)
        df_occ_year = get_occ_year_data(df_select, occ, year)
        # print length of df_occ_year
        print("df length:", len(df_occ_year))
        model = run_logit_model(df_occ_year, dependent = benefit, predictor = 'AI ROLE', cat_controls = [education, experience], ref_category = {education: "No Education Listed", experience: 'None Listed'})
        models_dict[(benefit, occ, year)] = model
        

LEAVE


  0%|          | 0/90 [00:00<?, ?it/s]

Architecture and Engineering Occupations 2019


  1%|          | 1/90 [00:00<01:18,  1.14it/s]

df length: 42432
Optimization terminated successfully.
         Current function value: 0.355468
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                  LEAVE   No. Observations:                42432
Model:                          Logit   Df Residuals:                    42420
Method:                           MLE   Df Model:                           11
Date:                Tue, 15 Oct 2024   Pseudo R-squ.:                0.006437
Time:                        23:53:45   Log-Likelihood:                -15083.
converged:                       True   LL-Null:                       -15181.
Covariance Type:            nonrobust   LLR p-value:                 6.605e-36
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                           -2.0735      0.032    -

...

                           Logit Regression Results                           
Dep. Variable:                CULTURE   No. Observations:               141154
Model:                          Logit   Df Residuals:                   141142
Method:                           MLE   Df Model:                           11
Date:                Tue, 15 Oct 2024   Pseudo R-squ.:                 0.01140
Time:                        23:58:31   Log-Likelihood:                -13800.
converged:                       True   LL-Null:                       -13959.
Covariance Type:            nonrobust   LLR p-value:                 1.302e-61
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                           -3.9566      0.027   -147.694      0.000      -4.009      -3.904
AI ROLE                          0.5486      0.735      0.746      0.456     

 99%|█████████▉| 89/90 [01:10<00:00,  1.13it/s]

                           Logit Regression Results                           
Dep. Variable:                CULTURE   No. Observations:               111268
Model:                          Logit   Df Residuals:                   111256
Method:                           MLE   Df Model:                           11
Date:                Tue, 15 Oct 2024   Pseudo R-squ.:                 0.03633
Time:                        23:58:32   Log-Likelihood:                -7453.1
converged:                       True   LL-Null:                       -7734.1
Covariance Type:            nonrobust   LLR p-value:                1.857e-113
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                           -4.8568      0.041   -118.830      0.000      -4.937      -4.777
AI ROLE                          0.8258      0.738      1.119      0.263     

100%|██████████| 90/90 [01:11<00:00,  1.27it/s]

df length: 110719
Optimization terminated successfully.
         Current function value: 0.291323
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                CULTURE   No. Observations:               110719
Model:                          Logit   Df Residuals:                   110707
Method:                           MLE   Df Model:                           11
Date:                Tue, 15 Oct 2024   Pseudo R-squ.:                0.007299
Time:                        23:58:32   Log-Likelihood:                -32255.
converged:                       True   LL-Null:                       -32492.
Covariance Type:            nonrobust   LLR p-value:                 9.485e-95
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                           -2.2622      0.014   -

In [40]:
import sys

# Estimate the size of the models dictionary in bytes
size_in_bytes = sys.getsizeof(models_dict)
size_in_megabytes = size_in_bytes / (1024 ** 2)  # Convert to MB
print(f"Size of models_dict: {size_in_megabytes:.2f} MB")

Size of models_dict: 0.02 MB


In [41]:
models_dict

{('LEAVE',
  'Architecture and Engineering Occupations',
  np.int32(2019)): <statsmodels.discrete.discrete_model.BinaryResultsWrapper at 0x7f1e80adb170>,
 ('LEAVE',
  'Architecture and Engineering Occupations',
  np.int32(2021)): <statsmodels.discrete.discrete_model.BinaryResultsWrapper at 0x7f102ceedbe0>,
 ('LEAVE',
  'Architecture and Engineering Occupations',
  np.int32(2022)): <statsmodels.discrete.discrete_model.BinaryResultsWrapper at 0x7f102ceeff20>,
 ('LEAVE',
  'Architecture and Engineering Occupations',
  np.int32(2020)): <statsmodels.discrete.discrete_model.BinaryResultsWrapper at 0x7f102ceec410>,
 ('LEAVE',
  'Architecture and Engineering Occupations',
  np.int32(2018)): <statsmodels.discrete.discrete_model.BinaryResultsWrapper at 0x7f1d17290dd0>,
 ('LEAVE',
  'Architecture and Engineering Occupations',
  np.int32(2023)): <statsmodels.discrete.discrete_model.BinaryResultsWrapper at 0x7f102d0cc740>,
 ('LEAVE',
  'Arts, Design, Entertainment, Sports, and Media Occupations',
 

In [42]:
# export models_dict to pickle
import pickle
with open('models_occ_year_dict.pickle', 'wb') as f:
    pickle.dump(models_dict, f)

In [4]:
import pickle

In [5]:
# import models_dict from pickle
with open('../exports/models_occ_year_dict.pickle', 'rb') as f:
    models_dict = pickle.load(f)

In [8]:
models_dict.keys()

dict_keys([('LEAVE', 'Architecture and Engineering Occupations', np.int32(2019)), ('LEAVE', 'Architecture and Engineering Occupations', np.int32(2021)), ('LEAVE', 'Architecture and Engineering Occupations', np.int32(2022)), ('LEAVE', 'Architecture and Engineering Occupations', np.int32(2020)), ('LEAVE', 'Architecture and Engineering Occupations', np.int32(2018)), ('LEAVE', 'Architecture and Engineering Occupations', np.int32(2023)), ('LEAVE', 'Arts, Design, Entertainment, Sports, and Media Occupations', np.int32(2019)), ('LEAVE', 'Arts, Design, Entertainment, Sports, and Media Occupations', np.int32(2021)), ('LEAVE', 'Arts, Design, Entertainment, Sports, and Media Occupations', np.int32(2022)), ('LEAVE', 'Arts, Design, Entertainment, Sports, and Media Occupations', np.int32(2020)), ('LEAVE', 'Arts, Design, Entertainment, Sports, and Media Occupations', np.int32(2018)), ('LEAVE', 'Arts, Design, Entertainment, Sports, and Media Occupations', np.int32(2023)), ('LEAVE', 'Business and Finan

In [11]:
list(models_dict.keys())[0][0]

'LEAVE'

In [20]:
coefficients = []
errors = []
pvalues = []
benefits = []
occupations = []
years = []

keys_list = list(models_dict.keys())
for key, value in models_dict.items(): 
    model = value
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
    except:
        coef = None
        err = None
        pvalue = None
    benefit = key[0]
    occ = key[1]
    year = key[2]
    
    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)
    benefits.append(benefit)
    occupations.append(occ)
    years.append(year)

# Creating DataFrame
results = {
    'Benefit': benefits,
    'Occupation': occupations,
    'Year': years,
    'AI Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues,
}

results_df = pd.DataFrame(results)

In [21]:
results_df

,Benefit,Occupation,Year,AI Coefficient,Error,P-Value
0,LEAVE,Architecture and Engineering Occupations,2019,-0.301721,0.125739,0.016
1,LEAVE,Architecture and Engineering Occupations,2021,-0.624508,0.104244,0.000
2,LEAVE,Architecture and Engineering Occupations,2022,-0.412328,0.075161,0.000
3,LEAVE,Architecture and Engineering Occupations,2020,-0.516013,0.134353,0.000
4,LEAVE,Architecture and Engineering Occupations,2018,-0.299582,0.164799,0.069
...,...,...,...,...,...,...
355,CULTURE,Transportation and Material Moving Occupations,2021,1.845892,0.207550,0.000
356,CULTURE,Transportation and Material Moving Occupations,2022,1.351989,0.208825,0.000
357,CULTURE,Transportation and Material Moving Occupations,2020,0.548588,0.735181,0.456
358,CULTURE,Transportation and Material Moving Occupations,2018,0.825759,0.737997,0.263
